In [1]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
# dataset & dataloader
from torch.utils.data import DataLoader
import torchvision.transforms as transforms # transforms is a process of image

# image => scale(0,1)=> normalize =>(-1,1)
transform = transforms.Compose([
    transforms.ToTensor(),# the data is authomatically convert of tensor & also scaled
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5)) # the set of  standard devition , mean value
])


trainset = CIFAR10(root = "./data",train = True, download = True,transform = transform)
testset = CIFAR10(root = "./data",train = False,download = True , transform = transform)

Files already downloaded and verified
Files already downloaded and verified


In [3]:
trainset

Dataset CIFAR10
    Number of datapoints: 50000
    Root location: ./data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
               Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
           )

In [4]:
trainloader = DataLoader(trainset,batch_size = 64, shuffle = True)
testloader = DataLoader(testset, batch_size = 64)

# Build the CNN 


In [15]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            # first layer of CNN
            nn.Conv2d(3,32,kernel_size=3,padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel  size = 2, stripe = 2

            # second layer of CNN
            nn.Conv2d(32,64,kernel_size=3,padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel  size = 2, stripe = 2

            # third  layer of CNN
            nn.Conv2d(64,128,kernel_size=3,padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), # kernel  size = 2, stripe = 2
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(4*4*128,256),
            nn.ReLU(),

            nn.Linear(256,10) # final output layer
        )

    def forward(self,x):
        x = self.conv_layers(x)
        x = x.view(x.size(0),-1) # flattening
        x = self.fc_layers(x)

        return x
        

In [16]:
model = CNN()

In [17]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

# Training the CNN

In [18]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model.forward(images) # FP

        loss = criterion(output,labels) # loss fnx
        loss.backward() #BP
        optimizer.step() # update params

        epoch_training_loss += loss.item()

    print(f"epoch = {epoch+1}/{epochs} & loss = {epoch_training_loss/len(trainloader)}")

epoch = 1/10 & loss = 1.357836206627014
epoch = 2/10 & loss = 0.9238224847389914
epoch = 3/10 & loss = 0.7465451607847458
epoch = 4/10 & loss = 0.6195672887670415
epoch = 5/10 & loss = 0.5100295592451949
epoch = 6/10 & loss = 0.42102015706355617
epoch = 7/10 & loss = 0.3312444455654877
epoch = 8/10 & loss = 0.25919462650861885
epoch = 9/10 & loss = 0.20251672450557848
epoch = 10/10 & loss = 0.15907475289405154


In [20]:
# Evaludate our CNN

correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images,labels in testloader:
        outputs = model.forward(images)
        _, predicted = torch.max(outputs,1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy = {correct_labels/ total_labels*100}")

Accuracy = 74.83
